# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 6 周：大数据与短期决策基础

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 理解大数据 5V 特征与三种数据类型
2. 掌握相关成本三条件，识别沉没成本陷阱
3. 掌握限制因素（瓶颈）下的最优生产决策：单位限制因素贡献毛益排序法
4. 用 Python 自动化限制因素优化并可视化

## 1. 大数据 5V 速览

| V | 含义 |
|---|---|
| Volume | 体量巨大（TB→PB 级）|
| Variety | 结构化 / 半结构化 / 非结构化（占 80%+）|
| Velocity | 实时产生与处理 |
| Value | 价值密度低，需算法提炼 |
| Veracity | 质量参差，数据清洗关键 |

In [ ]:
import json                                       # Python 内置 JSON 库（处理半结构化数据）
import numpy as np                                # 数值计算
import pandas as pd                               # 表格处理
import matplotlib.pyplot as plt                   # 绘图

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False        # 负号显示

In [ ]:
# ===== 演示处理三种数据类型 =====

# 1) 结构化数据：二维表（pandas 天然支持）
structured = pd.DataFrame({                       # 创建订单表
    '订单号': ['D001', 'D002', 'D003'],           # 订单编号列
    '客户': ['张三', '李四', '王五'],              # 客户列
    '金额': [299.0, 158.0, 899.0],               # 金额列
})
print('结构化数据（表格）:')                        # 标题
print(structured)                                 # 显示

# 2) 半结构化数据：JSON（键值对 + 嵌套）
json_str = '{"user_id": "U001", "action": "click", "item": "product_A", "amount": 299}'   # JSON 字符串
semi = json.loads(json_str)                       # loads 解析 JSON 字符串为字典
print('\n半结构化数据（JSON）:', semi)              # 显示解析结果

# 3) 非结构化数据：文本（简单词频统计）
text = '产品质量很好 物流速度快 客服态度一般 质量很好'   # 一段评论文本
words = text.split()                              # 按空格分词
freq = pd.Series(words).value_counts()            # 统计词频
print('\n非结构化数据（文本词频）:')                  # 标题
print(freq)                                       # 显示词频

## 2. 相关成本原则

相关成本必须同时满足三条件：**① 未来发生；② 因决策而不同；③ 现金流**。

**沉没成本不是相关成本**：50 元电影票已花掉，决策依据应是「接下来的时间值不值」，而不是「已经花了 50 元」。

## 3. 限制因素决策：XYZ 公司案例

三种产品共用 3000 机器工时。**决策关键**：不看单位利润，看**单位瓶颈资源贡献毛益**（$CM \div 单耗$）。

In [ ]:
df = pd.DataFrame({                               # 构建产品数据表
    '产品': ['A', 'B', 'C'],                       # 产品名
    '售价': [100, 150, 200],                       # 单价（元/件）
    '变动成本': [60, 90, 120],                     # 单位变动成本（元/件）
    '机器工时': [2, 5, 8],                         # 单位产品消耗工时（小时/件）
    '最大需求': [500, 300, 200],                   # 市场最大需求（件）
})

df['单位贡献毛益'] = df['售价'] - df['变动成本']     # CM = P - VC
df['单位工时贡献'] = df['单位贡献毛益'] / df['机器工时']   # 每小时贡献 = CM / 单耗

df_sorted = df.sort_values('单位工时贡献', ascending=False).reset_index(drop=True)   # 按每小时贡献降序排序
df_sorted[['产品', '单位贡献毛益', '机器工时', '单位工时贡献']]   # 显示关键列

In [ ]:
print('关键发现：')                                 # 结论预告
print('产品C 单位利润最高(80元)但每小时只贡献10元')   # 直觉陷阱
print('产品A 单位利润最低(40元)但每小时贡献20元 → 优先生产A')   # 正确决策

In [ ]:
# ===== 按排序依次分配 3000 工时 =====
total_hours = 3000                                # 瓶颈资源总量
remaining = total_hours                           # 剩余可用工时
results = []                                      # 收集各产品分配结果

for i, row in df_sorted.iterrows():               # 按每小时贡献从高到低遍历
    max_possible = min(row['最大需求'], int(remaining // row['机器工时']))   # 受需求和剩余工时双重限制
    used = max_possible * row['机器工时']          # 该产品实际消耗的工时
    contribution = max_possible * row['单位贡献毛益']   # 该产品创造的总贡献毛益
    results.append({                               # 记录分配方案
        '排序': i + 1, '产品': row['产品'],        # 优先级和产品名
        '计划产量': max_possible,                  # 实际安排的产量
        '使用工时': used, '贡献毛益': contribution   # 消耗工时与贡献
    })
    remaining -= used                             # 扣减剩余工时

result_df = pd.DataFrame(results)                 # 转为表格
print(result_df)                                  # 显示分配方案
print(f'\n总贡献毛益 = {result_df["贡献毛益"].sum():,.0f} 元')   # 合计 42,960 元
print(f'剩余未用工时 = {remaining} 小时')           # 应为 4 小时（500-62×8 不整除）

In [ ]:
# ===== 可视化最优组合 =====
fig, axes = plt.subplots(1, 2, figsize=(13, 5))   # 1 行 2 列子图
colors = ['#C49A6C', '#8B6F4E', '#E8D5C4']        # 三种产品颜色

axes[0].bar(result_df['产品'], result_df['计划产量'], color=colors, edgecolor='white')   # 柱状图：产量组合
axes[0].set_title('最优生产组合')                   # 子图标题
axes[0].set_ylabel('产量（件）')                    # y 轴
for i, v in enumerate(result_df['计划产量']):      # 在柱顶标注数值
    axes[0].text(i, v + 8, str(v), ha='center', fontweight='bold')   # 文字标注

axes[1].pie(result_df['贡献毛益'],                 # 饼图：各产品贡献占比
            labels=result_df['产品'],              # 标签
            autopct='%1.1f%%',                    # 显示百分比（1 位小数）
            colors=colors, startangle=90)          # 颜色与起始角
axes[1].set_title('各产品贡献毛益占比')             # 子图标题

plt.tight_layout()                                # 调整布局
plt.show()                                        # 显示

## 4. 封装通用优化函数

In [ ]:
def optimize_production(df_products, bottleneck_hours, bottleneck_col):
    """通用限制因素优化函数
    df_products: 含 单位贡献毛益/最大需求 的产品表
    bottleneck_hours: 瓶颈资源总量
    bottleneck_col: 瓶颈资源列名（如 机器工时/人工工时/材料用量）"""
    df = df_products.copy()                        # 复制避免修改原表
    df['单位瓶颈贡献'] = df['单位贡献毛益'] / df[bottleneck_col]   # 计算单位瓶颈资源贡献
    df = df.sort_values('单位瓶颈贡献', ascending=False)   # 降序排序
    remaining = bottleneck_hours                   # 剩余资源
    plan = []                                      # 计划列表
    for _, row in df.iterrows():                   # 依序分配
        make = min(row['最大需求'], int(remaining // row[bottleneck_col]))   # 受需求与剩余资源限制
        used = make * row[bottleneck_col]          # 消耗资源
        plan.append({'产品': row['产品'], '产量': make,   # 记录产量
                     '使用资源': used, '贡献毛益': make * row['单位贡献毛益']})   # 记录贡献
        remaining -= used                         # 扣减
    return pd.DataFrame(plan)                      # 返回方案表

# ===== 用 4 产品数据测试通用函数 =====
products = pd.DataFrame({                         # A/B/C/D 四产品
    '产品': ['A', 'B', 'C', 'D'],
    '单位贡献毛益': [40, 60, 80, 25],
    '机器工时': [2, 5, 8, 1],
    '最大需求': [500, 300, 200, 800],
})
optimize_production(products, 3000, '机器工时')    # 调用：3000 工时，瓶颈=机器工时

## 5. 练习：材料限制下的决策

原材料每月最多 1200kg。X：售价 80、变动 50、耗料 4kg/件、需求 200；Y：售价 120、变动 70、耗料 6kg/件、需求 150。  
**提示**：单位材料贡献 X = 30/4 = 7.5，Y = 50/6 ≈ 8.33 → 优先生产 Y。请用上面的函数验证总贡献毛益。

## 6. 本周小结

- 相关成本三条件：未来 + 因决策而异 + 现金流；沉没成本永远不相关
- 瓶颈决策看**单位限制因素贡献毛益**（CM ÷ 单耗），不是单位利润
- 五步法：找瓶颈 → 算 CM → 算单位瓶颈贡献 → 排序分配 → 算总贡献
- 封装成函数后，换任何瓶颈列（人工/材料）都能一键优化